# Lab 07-05 — EvaluationHarness A/B: which embedding model should you ship?

**Track 07 · Evaluation** — the payoff: one harness, full RAGAS quartet, clean attribution.

Labs 01-04 measured pieces (retrieval metrics, faithfulness, judge agreement, regression). This lab runs the full generation-metric suite — the RAGAS quartet faithfulness / answer relevance / context precision / context recall — through a hand-rolled harness on a hand-checked golden set, for TWO retrieval variants:

* **Variant A** — BGE (BAAI/bge-base-en-v1.5), the repo default embedder.
* **Variant B** — E5 (intfloat/multilingual-e5-base), the alternative embedder.

Everything else is identical (same corpus, same questions, same references, same judge, same generator, same top_k). Only the embedding model changes, so any metric difference is attributable to the embedder — that is the A/B discipline: change one knob, measure everything.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, faiss, and Groq directly — no repo component library. Every block of the pipeline is built right here: the BGE and E5 embedders, the two FAISS indexes, the retrievers, the generator, the four RAGAS-style metrics, and the harness that runs and aggregates them — all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

The golden set below is authored like `src/evaluation/golden.py`: hand-written references, but grounded — each question was kept only because its gold answer is contained verbatim in a retrievable passage of the indexed corpus (checked at authoring time), so a good retriever can genuinely answer it.

The pipeline, drawn inline:

```text
rag-mini first 800 passages
  -> retriever A: BGE  (default)
  -> retriever B: E5   (multilingual-e5-base)
  -> 6 golden questions, top_k = 3
  -> hand-rolled harness -> RAGAS quartet per variant
  -> aggregate means + verification gate (--verify)
```


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet`), already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the judge (and the generator it wraps) runs on Groq's hosted Llama model; the imports cell loads the key via python-dotenv.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-groq`, `sentence-transformers`, `faiss-cpu`, `pandas`, and `python-dotenv`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   sentence-transformers -> local BGE + E5 embeddings (HuggingFaceEmbeddings)
#   langchain-huggingface -> the HuggingFaceEmbeddings wrapper
#   langchain-community   -> the FAISS vector store
#   faiss-cpu             -> the FAISS index
#   langchain-groq        -> ChatGroq (judge + generator)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages.parquet corpus
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import pandas as pd  # noqa: E402  (reads the parquet corpus)

# LangChain + sentence-transformers + faiss + Groq — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from dotenv import load_dotenv  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)
load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

The golden set is **6 hand-authored questions**, each validated at authoring time so the gold answer is contained verbatim in one of the first 800 passages (the corpus is capped to keep the A/B fast). `TOP_K = 3` context chunks per question; the two embedding models are named constants. `GROQ_MODEL_NAME` selects the hosted judge/generator (the same model the lab's `GroqLLM` defaults to).


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
RAG_MINI = Path("Data/corpus/rag-mini-wikipedia")
PASSAGES_PATH = RAG_MINI / "passages.parquet"
N_PASSAGES = 800  # deterministic head of the corpus (keeps runtime low)
TOP_K = 3  # context chunks fed to the generator (harness.top_k)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
E5_MODEL_NAME = "intfloat/multilingual-e5-base"
GROQ_MODEL_NAME = "llama-3.3-70b-versatile"  # judge + generator (lab default)

# The golden set: hand-authored references, validated at authoring time so the
# gold answer appears verbatim in a retrievable passage of the corpus.
GOLDEN_QA: list[dict] = [
    {
        "doc": "rag-mini",
        "question": "Who assassinated Lincoln?",
        "reference": "John Wilkes Booth, Lincoln's assassin, can be seen in "
        "the crowd at Lincoln's second inauguration on March 4, 1865.",
    },
    {
        "doc": "rag-mini",
        "question": "The Celsius crater on the Moon is what?",
        "reference": "The Celsius crater on the Moon is named after the "
        "scientist Anders Celsius.",
    },
    {
        "doc": "rag-mini",
        "question": "What period of rapid economic growth did the United "
        "States experience during Coolidge's presidency?",
        "reference": "During Coolidge's presidency the United States "
        "experienced the period of rapid economic growth known as the "
        "Roaring Twenties.",
    },
    {
        "doc": "rag-mini",
        "question": "In 1905 Coolidge met and married whom?",
        "reference": "In 1905 Coolidge met and married Grace Anna Goodhue, "
        "a local schoolteacher and fellow Vermonter.",
    },
    {
        "doc": "rag-mini",
        "question": "When did Islam become the dominant religion in Java "
        "and Sumatra?",
        "reference": "Islam became the dominant religion in Java and "
        "Sumatra by the end of the 16th century.",
    },
    {
        "doc": "rag-mini",
        "question": "Who was on the committee with Adams to draft a "
        "Declaration of Independence?",
        "reference": "Adams was appointed on a committee with Thomas "
        "Jefferson, Benjamin Franklin, Robert R. Livingston and Roger "
        "Sherman to draft a Declaration of Independence.",
    },
]


## 2. Build one retriever per embedding model

`build_retriever(embedder, passages)` embeds the 800 passages, builds a FAISS index through a precomputed-vector passthrough (so the embed step stays separate), and returns a top-k retriever. The E5 embedder is wrapped inline to apply the E5 instruction prefixes (`query: ` / `passage: `) — the same behavior the lab's `E5Embedding` wrapper has; without the prefix, E5 underperforms its own training recipe. Two calls, two retrievers; nothing else differs.


In [ ]:
# --------------------------------------------------------------------------
# 2. Build one retriever per embedding model
# --------------------------------------------------------------------------
class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call. embed_query is delegated to the real embedder so
    the store's retriever can embed queries.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]],
                 query_embedder: Embeddings):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))
        self._query_embedder = query_embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._query_embedder.embed_query(text)


class InlineE5Embeddings(Embeddings):
    """E5 embedder with the instruction prefixes (hand-rolled).

    E5 models are trained with instruction prefixes: queries are prefixed
    "query: " and passages "passage: " before embedding. Mirrors the lab's
    E5Embedding wrapper — the prefix is the whole difference between E5 and
    BGE in this lab.
    """

    def __init__(self, model_name: str = E5_MODEL_NAME, **kwargs):
        self._emb = HuggingFaceEmbeddings(model_name=model_name, **kwargs)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._emb.embed_documents([f"passage: {t}" for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return self._emb.embed_query(f"query: {text}")


def build_retriever(embedder, passages: list[str]):
    """Index the passages and return a top-k similarity retriever."""
    vectors = embedder.embed_documents(passages)
    chunks = [
        Document(page_content=t, metadata={"id": i})
        for i, t in enumerate(passages)
    ]
    store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings(passages, vectors, embedder)
    )
    return store.as_retriever(search_kwargs={"k": TOP_K})


## 3. The harness + RAGAS quartet — hand-rolled inline

The lab imports `EvaluationHarness` and the four metrics from the shared evaluation block. Here all five are built by hand with the **same rubric and scale**:

* **faithfulness** — claims in the answer supported by the context (one judge call, `{"claims": [...], "supported": [bool, ...]}`).
* **answer_relevance** — embedding cosine between the question and the answer (no LLM — pure geometry).
* **context_precision** — the judge labels each retrieved chunk relevant/not in ONE call (`{"relevant": [bool, ...]}`); precision@k = mean of (relevant_count_at_k / k) over k where chunk k is relevant (RAGAS definition).
* **context_recall** — the judge checks whether each claim of the reference answer appears in the context (`{"present": [bool, ...]}`); score = present / total.

`InlineJudge` wraps `ChatGroq` — `ask()` generates the answer, `judge()` parses JSON (one retry, then `{"error": ...}`), `embed()` returns local BGE vectors. `InlineHarness.run` retrieves, generates, and scores every golden question; `aggregate` returns overall + per-doc means; `print_table` prints the aligned rows. The formulas — and the fact that three of the four metrics are judge calls — are the point of the cell.


In [ ]:
# --------------------------------------------------------------------------
# 3. The harness + RAGAS quartet — hand-rolled inline (same rubric)
# --------------------------------------------------------------------------
def _strip_code_fence(text: str) -> str:
    """Remove a surrounding markdown code fence (```json ... ```)."""
    lines = text.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


class InlineJudge:
    """LLM-as-judge over ChatGroq + local BGE embeddings (hand-rolled).

    Mirrors the shared LLMJudge contract the lab uses: ``ask()`` generates a
    plain-text answer, ``judge()`` returns the parsed JSON dict (one retry on
    a JSON failure, then {"error": ...}), ``embed()`` returns local BGE
    vectors.
    """

    def __init__(self, llm: ChatGroq, embedder):
        self.llm = llm
        self.embedder = embedder

    def ask(self, question: str, context: str) -> str:
        """Return a plain-text answer to ``question`` given ``context``."""
        prompt = (
            f"Context:\n{context}\n\nQuestion: {question}\n\n"
            "Answer concisely using only the context:"
        )
        return self.llm.invoke(prompt).content.strip()

    def judge(self, instruction: str, prompt: str) -> dict:
        last_error = ""
        for attempt in range(2):
            full = f"{instruction}\n\n{prompt}\n\nRespond with ONLY a valid JSON object."
            if attempt == 1:
                full += " Respond with ONLY valid JSON."
            try:
                text = self.llm.invoke(full).content
                return json.loads(_strip_code_fence(text))
            except (json.JSONDecodeError, ValueError) as exc:
                last_error = str(exc)
        return {"error": f"inline judge: could not parse JSON after 2 attempts: {last_error}"}

    def embed(self, texts: list[str]) -> list[list[float]]:
        return self.embedder.embed_documents(texts)


class InlineFaithfulnessMetric:
    """Faithfulness: fraction of the answer's claims supported by context."""

    def __init__(self, judge):
        self.judge = judge

    def score(self, question: str, context: str, answer: str) -> float:
        instruction = (
            "You are a faithfulness judge. Break the answer into atomic "
            "claims, then mark each claim as supported (true) or not (false) "
            "by the context."
        )
        prompt = (
            f"Context:\n{context}\n\nAnswer:\n{answer}\n\n"
            'Return JSON: {"claims": ["..."], "supported": [true, false, ...]}'
        )
        result = self.judge.judge(instruction, prompt)
        if "error" in result:
            return 0.0
        claims, supported = self._normalize(result)
        n = min(len(claims), len(supported))
        if n == 0:
            return 0.0
        return sum(1 for flag in supported[:n] if flag) / n

    @staticmethod
    def _normalize(result: dict) -> tuple[list[str], list[bool]]:
        """Coerce the judge's JSON into parallel (claims, supported) lists."""
        claims = result.get("claims")
        if not claims:
            claim = result.get("claim")
            claims = [claim] if claim else []
        supported = result.get("supported")
        if isinstance(supported, bool):
            supported = [supported] * len(claims)
        elif not isinstance(supported, list):
            supported = []
        return list(claims), list(supported)


class InlineAnswerRelevanceMetric:
    """Answer relevance: embedding cosine between question and answer.

    Short exact answers score low — that is expected and discussed in the
    lab: a terse "610.00" shares little token overlap with the question's
    embedding.
    """

    def __init__(self, judge):
        self.judge = judge

    @staticmethod
    def _cosine(a: list[float], b: list[float]) -> float:
        """Cosine similarity between two vectors (0.0 if either is zero)."""
        dot = sum(x * y for x, y in zip(a, b))
        na = sum(x * x for x in a) ** 0.5
        nb = sum(x * x for x in b) ** 0.5
        if na == 0.0 or nb == 0.0:
            return 0.0
        return dot / (na * nb)

    def score(self, question: str, answer: str) -> float:
        """Return the cosine similarity between question and answer embeddings."""
        q = self.judge.embed([question])[0]
        a = self.judge.embed([answer])[0]
        return self._cosine(q, a)


class InlineContextPrecisionMetric:
    """Context precision: RAGAS precision@k over the ranked chunks."""

    def __init__(self, judge):
        self.judge = judge

    def score(self, question: str, context: list) -> float:
        instruction = (
            "You are a retrieval judge. For each context chunk, decide whether "
            "it is relevant to answering the question."
        )
        numbered = "\n".join(f"{i + 1}. {chunk}" for i, chunk in enumerate(context))
        prompt = (
            f"Question: {question}\n\nContext chunks:\n{numbered}\n\n"
            'Return JSON: {"relevant": [true, false, ...]}'
        )
        result = self.judge.judge(instruction, prompt)
        if "error" in result:
            return 0.0
        relevant = result.get("relevant", [])
        if not relevant:
            return 0.0
        # RAGAS precision@k: mean over k where chunk k is relevant of
        # (relevant_count_at_k / k).
        scores = []
        count = 0
        for k, flag in enumerate(relevant, start=1):
            if flag:
                count += 1
                scores.append(count / k)
        if not scores:
            return 0.0
        return sum(scores) / len(scores)


class InlineContextRecallMetric:
    """Context recall: fraction of the reference's claims present in context."""

    def __init__(self, judge):
        self.judge = judge

    def score(self, question: str, context: str, reference_answer: str) -> float:
        instruction = (
            "You are a recall judge. Break the reference answer into atomic "
            "claims, then mark each claim present (true) or absent (false) in "
            "the context."
        )
        prompt = (
            f"Context:\n{context}\n\nReference answer:\n{reference_answer}\n\n"
            'Return JSON: {"present": [true, false, ...]}'
        )
        result = self.judge.judge(instruction, prompt)
        if "error" in result:
            return 0.0
        present = result.get("present", [])
        if not present:
            return 0.0
        return sum(1 for flag in present if flag) / len(present)


class InlineMetricResult:
    """One evaluated question: doc, question, reference, context, answer, 4 scores."""

    def __init__(self, doc, question, reference, context, answer,
                 faithfulness=0.0, answer_relevance=0.0,
                 context_precision=0.0, context_recall=0.0):
        self.doc = doc
        self.question = question
        self.reference = reference
        self.context = context
        self.answer = answer
        self.faithfulness = faithfulness
        self.answer_relevance = answer_relevance
        self.context_precision = context_precision
        self.context_recall = context_recall


class InlineHarness:
    """Run the four metrics over a golden question list (hand-rolled).

    Mirrors the shared EvaluationHarness contract: ``run`` retrieves,
    generates, and scores; ``aggregate`` returns overall + per-doc means;
    ``print_table`` prints an aligned table.
    """

    def __init__(self, judge, retriever, faithfulness_metric,
                 answer_relevance_metric, context_precision_metric,
                 context_recall_metric, top_k: int = 3):
        self.judge = judge
        self.retriever = retriever
        self.faithfulness_metric = faithfulness_metric
        self.answer_relevance_metric = answer_relevance_metric
        self.context_precision_metric = context_precision_metric
        self.context_recall_metric = context_recall_metric
        self.top_k = top_k

    @staticmethod
    def _to_text(chunks) -> str:
        """Join retrieved chunks (Documents or plain strings) into one text."""
        texts = []
        for chunk in chunks:
            text = getattr(chunk, "page_content", chunk)
            texts.append(text)
        return "\n\n".join(texts)

    def run(self, questions: list[dict]) -> list[InlineMetricResult]:
        """Evaluate every question: retrieve, generate, score."""
        results: list[InlineMetricResult] = []
        for q in questions:
            docs = self.retriever.invoke(q["question"])[: self.top_k]
            if not docs:
                raise RuntimeError(
                    f"InlineHarness: retriever returned no chunks for {q['question']!r}"
                )
            context = self._to_text(docs)
            answer = self.judge.ask(q["question"], context)
            results.append(
                InlineMetricResult(
                    doc=q["doc"],
                    question=q["question"],
                    reference=q["reference"],
                    context=context,
                    answer=answer,
                    faithfulness=self.faithfulness_metric.score(
                        q["question"], context, answer
                    ),
                    answer_relevance=self.answer_relevance_metric.score(
                        q["question"], answer
                    ),
                    context_precision=self.context_precision_metric.score(
                        q["question"], docs[: self.top_k]
                    ),
                    context_recall=self.context_recall_metric.score(
                        q["question"], context, q["reference"]
                    ),
                )
            )
        return results

    def aggregate(self, results: list[InlineMetricResult]) -> dict:
        """Return overall and per-doc mean scores."""
        metrics = [
            "faithfulness",
            "answer_relevance",
            "context_precision",
            "context_recall",
        ]

        def means(rows: list[InlineMetricResult]) -> dict:
            out: dict[str, float] = {}
            for name in metrics:
                values = [getattr(r, name) for r in rows]
                out[name] = sum(values) / len(values) if values else 0.0
            return out

        by_doc: dict[str, list[InlineMetricResult]] = {}
        for r in results:
            by_doc.setdefault(r.doc, []).append(r)
        return {
            "overall": means(results),
            "by_doc": {doc: means(rows) for doc, rows in by_doc.items()},
        }

    def print_table(self, results: list[InlineMetricResult]) -> None:
        """Print an aligned text table of every scored question."""
        header = f"{'doc':<20} {'question':<38} {'faith':>5} {'rel':>5} {'prec':>5} {'rec':>5} answer"
        print(header)
        print("-" * len(header))
        for r in results:
            answer = r.answer.replace("\n", " ")[:40]
            print(
                f"{r.doc:<20} {r.question[:38]:<38} "
                f"{r.faithfulness:>5.2f} {r.answer_relevance:>5.2f} "
                f"{r.context_precision:>5.2f} {r.context_recall:>5.2f} {answer}"
            )


## 4. Experiment — run the harness once per variant

The harness runs each variant end-to-end and scores every (question, context, answer) triple on the full RAGAS quartet. Retrieval quality shows up in context precision/recall; generation quality in faithfulness and answer relevance. `device="cpu"` on both embedders mirrors the lab: bulk-embedding on CPU keeps the run light on the shared machine's GPU. The thread cap below keeps the BLAS/OpenMP footprint small while several track agents share this box.


In [ ]:
# --------------------------------------------------------------------------
# 4. Experiment — run the harness once per variant
# --------------------------------------------------------------------------
# Several track agents share this machine — cap BLAS/OpenMP threads so the
# embedding step stays light on CPU and memory.
os.environ["OMP_NUM_THREADS"] = "2"
import torch  # noqa: E402
torch.set_num_threads(2)


def run_experiment() -> dict:
    df = pd.read_parquet(PASSAGES_PATH)
    passages = [str(r["passage"]).strip() for _, r in df.iterrows()][
        :N_PASSAGES
    ]

    # device="cpu": bulk-embedding on CPU avoids CUDA OOM and leaves the GPU
    # for the judge calls that follow.
    bge = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    e5 = InlineE5Embeddings(
        model_name=E5_MODEL_NAME,
        model_kwargs={"device": "cpu"},
    )

    t0 = time.perf_counter()
    retriever_bge = build_retriever(bge, passages)
    retriever_e5 = build_retriever(e5, passages)
    embed_s = time.perf_counter() - t0

    judge = InlineJudge(ChatGroq(model=GROQ_MODEL_NAME, temperature=0.0), bge)
    harness_kwargs = dict(
        judge=judge,
        faithfulness_metric=InlineFaithfulnessMetric(judge),
        answer_relevance_metric=InlineAnswerRelevanceMetric(judge),
        context_precision_metric=InlineContextPrecisionMetric(judge),
        context_recall_metric=InlineContextRecallMetric(judge),
        top_k=TOP_K,
    )

    t0 = time.perf_counter()
    results_bge = InlineHarness(
        retriever=retriever_bge, **harness_kwargs
    ).run(GOLDEN_QA)
    results_e5 = InlineHarness(
        retriever=retriever_e5, **harness_kwargs
    ).run(GOLDEN_QA)
    eval_s = time.perf_counter() - t0

    return {
        "passages": len(passages),
        "embed_s": embed_s,
        "eval_s": eval_s,
        "results_bge": results_bge,
        "results_e5": results_e5,
    }


## 5. Demo — print the artifact

The demo prints both variants' per-question rows and the A/B means table. Reading the rows beats reading the means: one bad retrieval on a single question explains a dip better than an average ever will. Expect BGE to win context_recall — the E5 miss on one passage is visible as a 0 on that row.


In [ ]:
# --------------------------------------------------------------------------
# 5. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 72)
    print("Lab 07-05 — EvaluationHarness A/B: BGE vs E5 embeddings")
    print(f"rag-mini {exp['passages']} passages, "
          f"{len(exp['results_bge'])} golden questions, top_k={TOP_K}")
    print("=" * 72)

    harness = InlineHarness(
        judge=None, retriever=None, faithfulness_metric=None,
        answer_relevance_metric=None, context_precision_metric=None,
        context_recall_metric=None, top_k=TOP_K,
    )
    agg_bge = harness.aggregate(exp["results_bge"])["overall"]
    agg_e5 = harness.aggregate(exp["results_e5"])["overall"]

    print(f"\n[1] Per-question rows — variant A (BGE):")
    harness.print_table(exp["results_bge"])
    print(f"\n[2] Per-question rows — variant B (E5):")
    harness.print_table(exp["results_e5"])

    print(f"\n[3] A/B means (RAGAS quartet):")
    print(f"    {'metric':<20} {'A (BGE)':>9} {'B (E5)':>9}  delta")
    for key in ("faithfulness", "answer_relevance",
                "context_precision", "context_recall"):
        diff = agg_bge[key] - agg_e5[key]
        print(f"    {key:<20} {agg_bge[key]:>9.3f} {agg_e5[key]:>9.3f}  "
              f"{diff:+.3f}")

    print(f"\n[4] Timing: embed {exp['embed_s']:.0f}s, harness both "
          f"{exp['eval_s']:.0f}s")

    print(f"\n[5] Takeaway")
    print("    A/B on embeddings: same corpus, same questions, same judge,")
    print("    same generator — only the embedder differs, so metric deltas")
    print("    are attributable. The harness gives the full RAGAS quartet")
    print("    instead of a single number: retrieval quality shows up in")
    print("    context precision/recall, generation quality in faithfulness")
    print("    and answer relevance. Reading the rows beats reading the")
    print("    means: one bad retrieval on a question explains a dip better")
    print("    than a mean ever will. Note the judge is the same model used")
    print("    for both variants — the comparison stays fair even though the")
    print("    judge is not interchangeable with a reference. This is the")
    print("    same harness you would wire into a golden-set regression")
    print("    (lab 04) for every embedding/retriever change.")


## 6. Verification gate

The gate enforces the A/B contract: both variants evaluate all 6 golden questions, every metric lands in [0, 1], both variants find the golden answers (context_recall > 0 — the golden set is answerable), and the A/B is meaningful — the variants differ on at least one metric. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 6. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    harness = InlineHarness(
        judge=None, retriever=None, faithfulness_metric=None,
        answer_relevance_metric=None, context_precision_metric=None,
        context_recall_metric=None, top_k=TOP_K,
    )
    agg_bge = harness.aggregate(exp["results_bge"])["overall"]
    agg_e5 = harness.aggregate(exp["results_e5"])["overall"]

    for label, results in (("BGE", exp["results_bge"]),
                           ("E5", exp["results_e5"])):
        checks.append(
            (f"{label}: {len(results)} questions evaluated (== 6)",
             len(results) == len(GOLDEN_QA)))
        for key in ("faithfulness", "answer_relevance",
                    "context_precision", "context_recall"):
            checks.append((f"{label} {key} in [0, 1]",
                           0.0 <= harness.aggregate(results)["overall"][key]
                           <= 1.0))

    # The golden set must be answerable by both variants (validated at
    # authoring time: gold contained in a retrievable passage).
    checks.append(("BGE context_recall > 0 (golden set is answerable)",
                   agg_bge["context_recall"] > 0.0))
    checks.append(("E5 context_recall > 0 (golden set is answerable)",
                   agg_e5["context_recall"] > 0.0))

    # The A/B is meaningful only if the variants differ somewhere.
    differs = any(abs(agg_bge[k] - agg_e5[k]) > 0.01
                  for k in ("faithfulness", "answer_relevance",
                            "context_precision", "context_recall"))
    checks.append(("A/B meaningful: variants differ on >= 1 metric", differs))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Embedding 800 passages twice (BGE + E5) on CPU takes a couple of minutes (no downloads — both models are cached); the 12 generations + 36 judge calls follow. `exp` holds both variants' results plus timing.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Both variants' per-question rows and the A/B means table. Reading the rows beats reading the means: one bad retrieval on a single question explains a dip better than an average ever will.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the rag-mini files are intact and `GROQ_API_KEY` is set.


In [ ]:
verify_gate(exp)
